# 🔥 Notebook 2: Handling Flash Sales

**The "Taylor Swift Problem" — what happens when millions of users try to buy tickets the instant they go on sale?**

---

## 🎯 Learning Objectives

By the end of this notebook you will understand:

1. **Why flash sales overwhelm systems** — what actually breaks when 10 million users hit "Buy" at the same time
2. **Virtual waiting queues** — the real-world pattern used by Ticketmaster, airlines, and concert platforms
3. **Redis sorted sets for fair ordering** — how `ZADD` + timestamps give you a perfect FIFO queue
4. **Controlled admission** — admitting users in small batches so the database stays healthy

> 💡 **Real-world context:** In November 2022, Ticketmaster's system crashed when 14 million users tried to buy Taylor Swift Eras Tour tickets simultaneously. The site was down for hours, tickets sold out in minutes, and Congress held hearings about it. This notebook teaches the engineering patterns that *prevent* that kind of disaster.

## 🛠️ Setup

### 1. Start the services

From the `system-designs/ticketmaster/` directory, run:

```bash
docker compose up -d
```

This starts:
- **PostgreSQL** on `localhost:5433` — our ticket database
- **Redis** on `localhost:6380` — our queue engine
- **Adminer** on [http://localhost:8081](http://localhost:8081) — database GUI
- **RedisInsight** on [http://localhost:5541](http://localhost:5541) — Redis GUI

### 2. Select the notebook kernel

This project has its own virtual environment. In VS Code:
1. Click the **kernel picker** (top-right of the notebook)
2. Choose the `.venv` kernel from `system-designs/ticketmaster/.venv`
3. If it doesn't appear, reload the window: `Cmd+Shift+P` → **"Reload Window"**

### 3. Inspect the data (optional)

- Open **Adminer** at [http://localhost:8081](http://localhost:8081) (System: PostgreSQL, Server: `ticketmaster-postgres`, User: `demo`, Password: `demo`, Database: `ticketmaster`)
- Open **RedisInsight** at [http://localhost:5541](http://localhost:5541) and connect to `ticketmaster-redis:6379`

In [ ]:
# ============================================================
# 📦 Imports & Configuration
# ============================================================

import psycopg2
import psycopg2.extras
import redis
import time
import json
import threading
import random
from tabulate import tabulate

# -- Connection config ------------------------------------------------
PG_CONFIG = {
    "host": "localhost",
    "port": 5433,
    "dbname": "ticketmaster",
    "user": "demo",
    "password": "demo",
}

REDIS_CONFIG = {
    "host": "localhost",
    "port": 6380,
    "decode_responses": True,
}

# -- Helpers -----------------------------------------------------------
def get_pg_connection():
    """Return a new PostgreSQL connection."""
    return psycopg2.connect(**PG_CONFIG)

def get_redis_client():
    """Return a Redis client."""
    return redis.Redis(**REDIS_CONFIG)

# -- Test connections --------------------------------------------------
conn = get_pg_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM tickets WHERE event_id = 1;")
total = cur.fetchone()[0]
cur.execute("SELECT COUNT(*) FROM tickets WHERE event_id = 1 AND status = 'available';")
available = cur.fetchone()[0]
cur.close()
conn.close()

r = get_redis_client()
r.ping()

print("✅ PostgreSQL connected")
print(f"   Event 1 ('The Eras Tour - NYC'): {total} total seats, {available} available")
print("✅ Redis connected")

---

## 💥 The Flash Sale Problem

Imagine this: Taylor Swift announces a new concert. Tickets go on sale at **10:00 AM sharp**.

At 9:59 AM, **10 million users** are sitting on the booking page, fingers hovering over the "Buy" button. The clock hits 10:00 and they ALL click at the same time.

### What happens without protection?

Every single user's browser sends a request to the server at the same instant. The server tries to:

1. **Query available seats** — 10 million `SELECT` queries hit the database simultaneously
2. **Lock a seat** — 10 million `SELECT FOR UPDATE` queries fight for the same rows
3. **Create a booking** — most of these fail because someone else already locked the seat

The result is chaos:
- 🔴 **Database crushed** — connection pool exhausted, queries timing out
- 🔴 **Most users get errors** — "Something went wrong, please try again"
- 🔴 **Seat map is chaotic** — seats appear and disappear as locks expire
- 🔴 **Response times spike** — pages take 30+ seconds to load (if they load at all)

### 🏪 The Store Analogy

Think of it like a **Black Friday sale**:

- **Without a queue:** 10,000 people rush through one tiny door at the same time. People get crushed, shelves get knocked over, nobody can actually shop.
- **With a queue:** Everyone lines up outside in an orderly line. A security guard lets in 20 people at a time. Everyone gets a fair shot, and the store stays intact.

That's exactly what we're going to build — a **virtual waiting queue** that turns a stampede into an orderly line.

Let's first **see the chaos** by simulating a flash sale without any protection. 👇

In [ ]:
# ============================================================
# 💥 Simulate a flash sale WITHOUT a queue
# ============================================================
# 50 users all try to book tickets for Event 1 at the same time.
# Each user picks a random available ticket and tries to book it
# using SELECT FOR UPDATE (the standard approach).
# ============================================================

EVENT_ID = 1
NUM_USERS = 50

# Track results from each thread
results = {"success": [], "failure": [], "times": []}
results_lock = threading.Lock()

def try_book_ticket(user_id):
    """One user tries to grab a random available ticket. No queue, no protection."""
    start = time.time()
    conn = None
    try:
        conn = get_pg_connection()
        conn.autocommit = False
        cur = conn.cursor()

        # Pick a random available ticket and lock it
        cur.execute("""
            SELECT id, section, row_label, seat_number, price
            FROM tickets
            WHERE event_id = %s AND status = 'available'
            ORDER BY random()
            LIMIT 1
            FOR UPDATE SKIP LOCKED;
        """, (EVENT_ID,))
        row = cur.fetchone()

        if not row:
            conn.rollback()
            with results_lock:
                results["failure"].append((user_id, "no_ticket"))
                results["times"].append(time.time() - start)
            return

        ticket_id, section, row_label, seat_num, price = row

        # Simulate some processing delay (payment validation, etc.)
        time.sleep(random.uniform(0.05, 0.2))

        # Mark ticket as sold
        cur.execute("UPDATE tickets SET status = 'sold' WHERE id = %s;", (ticket_id,))

        # Create a booking
        cur.execute("""
            INSERT INTO bookings (user_id, event_id, total_price, status)
            VALUES (%s, %s, %s, 'confirmed') RETURNING id;
        """, (user_id, EVENT_ID, float(price)))
        booking_id = cur.fetchone()[0]

        cur.execute("INSERT INTO booking_tickets (booking_id, ticket_id) VALUES (%s, %s);",
                    (booking_id, ticket_id))

        conn.commit()
        with results_lock:
            results["success"].append((user_id, ticket_id, section, row_label, seat_num))
            results["times"].append(time.time() - start)

    except Exception as e:
        if conn:
            conn.rollback()
        with results_lock:
            results["failure"].append((user_id, str(e)[:60]))
            results["times"].append(time.time() - start)
    finally:
        if conn:
            conn.close()

# -- Launch all threads at once (simulating the stampede) ----
print(f"💥 Launching {NUM_USERS} users simultaneously...\n")
threads = []
start_time = time.time()
for uid in range(1, NUM_USERS + 1):
    t = threading.Thread(target=try_book_ticket, args=(uid,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

total_time = time.time() - start_time

# -- Results --------------------------------------------------
print("📊 Flash Sale Results (NO queue)")
print("=" * 45)
print(f"  ✅ Successful bookings : {len(results['success'])}")
print(f"  ❌ Failed attempts     : {len(results['failure'])}")
print(f"  ⏱️  Total time          : {total_time:.2f}s")
print(f"  ⏱️  Avg response time   : {sum(results['times'])/len(results['times']):.3f}s")
print(f"  ⏱️  Max response time   : {max(results['times']):.3f}s")
print()

if results["failure"]:
    # Show a sample of failures
    fail_reasons = {}
    for uid, reason in results["failure"]:
        fail_reasons[reason] = fail_reasons.get(reason, 0) + 1
    print("❌ Failure breakdown:")
    for reason, count in fail_reasons.items():
        print(f"   {reason}: {count} users")

print()
print("⚠️  Notice: All users hit the database at once.")
print("   In production with 10M users, this would crash the server.")

In [ ]:
# ============================================================
# 🧹 Reset: undo the chaos so we have a clean slate
# ============================================================
# Delete the bookings we just created and set tickets back to 'available'.
# We keep the original 25 'sold' tickets from the seed data.

conn = get_pg_connection()
cur = conn.cursor()

# Delete booking_tickets and bookings created by the simulation
cur.execute("DELETE FROM booking_tickets;")
cur.execute("DELETE FROM bookings;")

# Reset all tickets for event 1 back to available, then re-sell the original 25
cur.execute("UPDATE tickets SET status = 'available' WHERE event_id = 1;")
cur.execute("""
    UPDATE tickets SET status = 'sold'
    WHERE id IN (
        SELECT id FROM tickets WHERE event_id = 1 ORDER BY id LIMIT 25
    );
""")

conn.commit()

# Verify
cur.execute("SELECT status, COUNT(*) FROM tickets WHERE event_id = 1 GROUP BY status ORDER BY status;")
rows = cur.fetchall()
cur.close()
conn.close()

print("🧹 Reset complete! Event 1 ticket status:")
print(tabulate(rows, headers=["Status", "Count"], tablefmt="simple_grid"))

---

## 🚪 Solution: Virtual Waiting Queue

Instead of letting everyone hit the booking page at once, we put users in a **virtual queue** and admit them in small batches.

### How it works

```
User arrives → Join Queue (Redis ZADD) → Wait for admission → Enter booking page → Book ticket
```

### Why Redis Sorted Sets?

A Redis **sorted set** (`ZSET`) is the perfect data structure for a waiting queue:

| Operation | Redis Command | Time Complexity | What it does |
|-----------|--------------|----------------|--------------|
| Join queue | `ZADD` | O(log N) | Add user with timestamp as score |
| Check position | `ZRANK` | O(log N) | Get user's position (0-indexed) |
| Queue size | `ZCARD` | O(1) | Total users waiting |
| Admit next batch | `ZPOPMIN` | O(log N × M) | Pop M users with lowest scores |

The **score = timestamp** trick gives us perfect **FIFO (First In, First Out)** ordering — whoever joined first gets the lowest score, so they get popped first.

### This is what real sites do

- **Ticketmaster** — "You are in the queue. Estimated wait: 12 minutes."
- **Airlines** — "Please wait, you will be redirected when it's your turn."
- **Supreme/Nike drops** — Virtual waiting rooms before product launches

Let's build it! 👇

In [ ]:
# ============================================================
# 🚪 Implement the virtual waiting queue
# ============================================================

def join_queue(event_id, user_id):
    """Add user to the waiting queue. Score = timestamp for FIFO order."""
    r = get_redis_client()
    score = time.time()
    r.zadd(f"queue:{event_id}", {f"user:{user_id}": score})
    position = r.zrank(f"queue:{event_id}", f"user:{user_id}")
    return position + 1  # 1-indexed for human readability

# -- Demo: 20 users join the queue for Event 1 ----------------
EVENT_ID = 1

# Clean any leftover queue data first
r = get_redis_client()
r.delete(f"queue:{EVENT_ID}")

print("🚪 Users joining the queue for 'The Eras Tour - NYC':\n")
rows = []
for uid in range(1, 21):
    pos = join_queue(EVENT_ID, uid)
    rows.append([f"User {uid}", f"#{pos}"])
    time.sleep(0.01)  # tiny delay so timestamps differ

print(tabulate(rows, headers=["User", "Position"], tablefmt="simple_grid"))
print(f"\n📊 Queue size: {r.zcard(f'queue:{EVENT_ID}')} users")

---

## ⏳ Checking Queue Position

Users need to know where they stand — nobody likes staring at a blank screen.

With sorted sets, `ZRANK` gives us the user's position in **O(log N)** time. Even with 10 million users in the queue, looking up your position is nearly instant.

We can also estimate wait time based on position and batch speed:

```
estimated_wait = (position / batch_size) × batch_interval_seconds
```

For example, if you're #500 in line, we admit 50 per batch, and each batch takes 10 seconds:
- `(500 / 50) × 10 = 100 seconds ≈ ~2 minutes`

In [ ]:
# ============================================================
# ⏳ Queue position & estimated wait time
# ============================================================

def get_queue_position(event_id, user_id):
    """Get a user's current position in the queue (1-indexed).
    Returns None if the user is not in the queue."""
    r = get_redis_client()
    rank = r.zrank(f"queue:{event_id}", f"user:{user_id}")
    if rank is None:
        return None
    return rank + 1

def get_queue_size(event_id):
    """Get the total number of users waiting in the queue."""
    r = get_redis_client()
    return r.zcard(f"queue:{event_id}")

def get_estimated_wait(event_id, user_id, batch_size=5, batch_interval_seconds=10):
    """Estimate how long a user will wait before being admitted.
    Returns estimated seconds, or None if user is not in the queue."""
    position = get_queue_position(event_id, user_id)
    if position is None:
        return None
    # How many batches until this user is admitted?
    batches_ahead = (position - 1) // batch_size
    return batches_ahead * batch_interval_seconds

# -- Demo: check positions for a few users --------------------
EVENT_ID = 1

print(f"📊 Queue size: {get_queue_size(EVENT_ID)} users\n")

sample_users = [1, 5, 10, 15, 20]
rows = []
for uid in sample_users:
    pos = get_queue_position(EVENT_ID, uid)
    wait = get_estimated_wait(EVENT_ID, uid, batch_size=5, batch_interval_seconds=10)
    rows.append([f"User {uid}", f"#{pos}", f"{wait}s" if wait is not None else "N/A"])

print(tabulate(rows, headers=["User", "Position", "Est. Wait"], tablefmt="simple_grid"))
print()
print("💡 User 1 is first — they'll be in the next batch (0s wait).")
print("   User 20 is last — they'll wait for several batches.")

---

## 🎟️ Admitting Users in Batches

Now for the key piece: **controlled admission**.

Instead of letting all 20 users flood the booking page, we:

1. **Pop N users** from the front of the queue (`ZPOPMIN`) — these are the ones who joined earliest
2. **Add them to an "admitted" set** with a **TTL** (time-to-live) — they have 10 minutes to complete their purchase
3. **Only admitted users** can access the booking page — everyone else sees "Please wait"

### Why TTL?

If an admitted user doesn't complete their purchase (they got distracted, their payment failed, etc.), the TTL ensures their admission expires automatically. The system then admits the next batch from the queue — no wasted spots.

```
Queue: [User1, User2, ..., User20]
          │
          ▼  ZPOPMIN(5)
Admitted: {User1, User2, User3, User4, User5}  ← 10-min TTL each
Queue:    [User6, User7, ..., User20]           ← still waiting
```

In [ ]:
# ============================================================
# 🎟️ Admitting users in batches
# ============================================================

def admit_batch(event_id, batch_size=5):
    """Admit the next batch_size users from the queue.
    Returns a list of admitted user keys."""
    r = get_redis_client()
    # Pop the first N users (lowest scores = earliest arrivals)
    admitted = r.zpopmin(f"queue:{event_id}", batch_size)
    admitted_users = []
    for user_key, score in admitted:
        # Give each admitted user a 10-minute window (600 seconds)
        r.setex(f"admitted:{event_id}:{user_key}", 600, "1")
        admitted_users.append(user_key)
    return admitted_users

def is_admitted(event_id, user_id):
    """Check if a user has been admitted to the booking page."""
    r = get_redis_client()
    return r.exists(f"admitted:{event_id}:user:{user_id}") == 1

# -- Demo: admit the first batch of 5 -------------------------
EVENT_ID = 1

print(f"📊 Before admission:  Queue size = {get_queue_size(EVENT_ID)}")
print()

admitted = admit_batch(EVENT_ID, batch_size=5)
print(f"✅ Admitted {len(admitted)} users: {admitted}")
print(f"📊 After admission:   Queue size = {get_queue_size(EVENT_ID)}")
print()

# Check who's admitted and who's still waiting
rows = []
for uid in range(1, 11):
    admitted_status = is_admitted(EVENT_ID, uid)
    queue_pos = get_queue_position(EVENT_ID, uid)
    if admitted_status:
        status = "✅ ADMITTED"
    elif queue_pos:
        status = f"⏳ Waiting (#{queue_pos})"
    else:
        status = "❌ Not in queue"
    rows.append([f"User {uid}", status])

print(tabulate(rows, headers=["User", "Status"], tablefmt="simple_grid"))
print()
print("💡 Users 1-5 can now access the booking page.")
print("   Users 6-10 are still waiting in the queue.")

---

## 🛡️ Protected Booking

Now that we have the queue, the booking endpoint adds a simple check:

```
1. Is this user admitted?  →  No?  →  "Please wait in the queue."
2. Yes?  →  Proceed with SELECT FOR UPDATE  →  Book the ticket.
```

This is the crucial piece that ties the queue to the database. Only a controlled number of users (the admitted batch) ever reach the database at the same time. The database stays healthy, and every admitted user gets a smooth experience.

In [ ]:
# ============================================================
# 🛡️ Protected booking — only admitted users can book
# ============================================================

def book_with_queue_check(event_id, user_id, ticket_id):
    """Book a ticket, but only if the user has been admitted through the queue.
    Returns (success: bool, message: str)."""

    # Step 1: Check admission
    if not is_admitted(event_id, user_id):
        return False, "❌ Not admitted — please wait in the queue."

    # Step 2: Proceed with the actual booking (protected by SELECT FOR UPDATE)
    conn = get_pg_connection()
    try:
        conn.autocommit = False
        cur = conn.cursor()

        # Lock the specific ticket
        cur.execute("""
            SELECT id, price FROM tickets
            WHERE id = %s AND event_id = %s AND status = 'available'
            FOR UPDATE;
        """, (ticket_id, event_id))
        row = cur.fetchone()

        if not row:
            conn.rollback()
            return False, "❌ Ticket not available (already sold or doesn't exist)."

        tid, price = row

        # Mark ticket as sold
        cur.execute("UPDATE tickets SET status = 'sold' WHERE id = %s;", (tid,))

        # Create booking
        cur.execute("""
            INSERT INTO bookings (user_id, event_id, total_price, status)
            VALUES (%s, %s, %s, 'confirmed') RETURNING id;
        """, (user_id, event_id, float(price)))
        booking_id = cur.fetchone()[0]

        cur.execute("INSERT INTO booking_tickets (booking_id, ticket_id) VALUES (%s, %s);",
                    (booking_id, tid))

        conn.commit()

        # Remove admission key (one booking per admission)
        r = get_redis_client()
        r.delete(f"admitted:{event_id}:user:{user_id}")

        return True, f"✅ Booked! Booking #{booking_id}, Ticket #{tid}, ${price}"

    except Exception as e:
        conn.rollback()
        return False, f"❌ Error: {str(e)[:80]}"
    finally:
        conn.close()

# -- Demo: admitted user vs non-admitted user ------------------
EVENT_ID = 1

# Get an available ticket to try booking
conn = get_pg_connection()
cur = conn.cursor()
cur.execute("SELECT id FROM tickets WHERE event_id = %s AND status = 'available' LIMIT 1;", (EVENT_ID,))
ticket_id = cur.fetchone()[0]
cur.close()
conn.close()

print("🛡️ Protected Booking Demo\n")

# User 3 was admitted earlier — should succeed
success, msg = book_with_queue_check(EVENT_ID, 3, ticket_id)
print(f"User 3 (admitted):     {msg}")

# User 15 is still in the queue — should be rejected
success, msg = book_with_queue_check(EVENT_ID, 15, ticket_id)
print(f"User 15 (not admitted): {msg}")

print()
print("💡 The queue acts as a gatekeeper — only admitted users touch the database.")

---

## 🏁 Full Simulation: Flash Sale with Queue

Let's put it all together and simulate a complete flash sale using the queue system:

1. **30 users** join the queue
2. We **admit batches of 5** every 2 seconds
3. Each admitted user **tries to book a random available ticket**
4. We track the **timeline** — queue size, admissions, bookings

Compare this to the chaos from earlier — you'll see the difference immediately.

In [ ]:
# ============================================================
# 🧹 Reset before the full simulation
# ============================================================

# Clean up Redis
r = get_redis_client()
for key in r.scan_iter("queue:*"):
    r.delete(key)
for key in r.scan_iter("admitted:*"):
    r.delete(key)

# Reset database
conn = get_pg_connection()
cur = conn.cursor()
cur.execute("DELETE FROM booking_tickets;")
cur.execute("DELETE FROM bookings;")
cur.execute("UPDATE tickets SET status = 'available' WHERE event_id = 1;")
cur.execute("""
    UPDATE tickets SET status = 'sold'
    WHERE id IN (
        SELECT id FROM tickets WHERE event_id = 1 ORDER BY id LIMIT 25
    );
""")
conn.commit()

cur.execute("SELECT COUNT(*) FROM tickets WHERE event_id = 1 AND status = 'available';")
available = cur.fetchone()[0]
cur.close()
conn.close()

print(f"🧹 Clean slate: {available} available tickets for Event 1")

In [ ]:
# ============================================================
# 🏁 Full simulation: flash sale WITH the queue system
# ============================================================

EVENT_ID = 1
NUM_USERS = 30
BATCH_SIZE = 5
BATCH_INTERVAL = 2  # seconds between batches

timeline = []       # log of events
bookings_made = []  # successful bookings
booking_failures = []  # failed booking attempts

sim_start = time.time()

def ts():
    """Seconds since simulation start."""
    return f"{time.time() - sim_start:.1f}s"

# -- Phase 1: Users join the queue (simulating the rush) ------
print("🏁 FLASH SALE SIMULATION")
print("=" * 50)
print(f"\n📋 Config: {NUM_USERS} users, batches of {BATCH_SIZE}, {BATCH_INTERVAL}s interval\n")

r = get_redis_client()
r.delete(f"queue:{EVENT_ID}")

print(f"[{ts()}] 🚪 {NUM_USERS} users joining the queue...")
for uid in range(1, NUM_USERS + 1):
    join_queue(EVENT_ID, uid)
    time.sleep(0.005)  # tiny stagger

queue_size = get_queue_size(EVENT_ID)
timeline.append((ts(), f"{queue_size} users in queue"))
print(f"[{ts()}] 📊 Queue size: {queue_size}")

# -- Phase 2: Admit and book in batches -----------------------
batch_num = 0
while get_queue_size(EVENT_ID) > 0:
    batch_num += 1
    time.sleep(BATCH_INTERVAL)

    admitted = admit_batch(EVENT_ID, BATCH_SIZE)
    timeline.append((ts(), f"Batch {batch_num}: admitted {admitted}"))
    print(f"[{ts()}] ✅ Batch {batch_num}: admitted {len(admitted)} users → {admitted}")

    # Each admitted user tries to book a random available ticket
    for user_key in admitted:
        # Extract user_id from "user:X"
        uid = int(user_key.split(":")[1])

        # Find a random available ticket
        conn = get_pg_connection()
        cur = conn.cursor()
        cur.execute("""
            SELECT id FROM tickets
            WHERE event_id = %s AND status = 'available'
            ORDER BY random() LIMIT 1;
        """, (EVENT_ID,))
        row = cur.fetchone()
        cur.close()
        conn.close()

        if row:
            ticket_id = row[0]
            success, msg = book_with_queue_check(EVENT_ID, uid, ticket_id)
            if success:
                bookings_made.append(uid)
            else:
                booking_failures.append((uid, msg))
        else:
            booking_failures.append((uid, "No tickets left"))

    remaining = get_queue_size(EVENT_ID)
    print(f"         📊 Queue remaining: {remaining} | Booked so far: {len(bookings_made)}")

total_time = time.time() - sim_start

# -- Results --------------------------------------------------
print(f"\n{'=' * 50}")
print("📊 SIMULATION RESULTS")
print(f"{'=' * 50}")
print(f"  👥 Total users          : {NUM_USERS}")
print(f"  ✅ Successful bookings  : {len(bookings_made)}")
print(f"  ❌ Failed bookings      : {len(booking_failures)}")
print(f"  🔄 Batches processed    : {batch_num}")
print(f"  ⏱️  Total time           : {total_time:.1f}s")

if booking_failures:
    print(f"\n  ❌ Failures:")
    for uid, reason in booking_failures:
        print(f"     User {uid}: {reason}")

# Check final ticket counts
conn = get_pg_connection()
cur = conn.cursor()
cur.execute("SELECT status, COUNT(*) FROM tickets WHERE event_id = 1 GROUP BY status ORDER BY status;")
rows = cur.fetchall()
cur.close()
conn.close()

print(f"\n📊 Final ticket status for Event 1:")
print(tabulate(rows, headers=["Status", "Count"], tablefmt="simple_grid"))

In [ ]:
# ============================================================
# 📊 Comparison: No Queue vs. With Queue
# ============================================================

comparison = [
    ["Database load",        "50 concurrent connections",        f"{BATCH_SIZE} concurrent connections"],
    ["User experience",      "Errors, timeouts, confusion",      "Clear position, estimated wait"],
    ["Booking fairness",     "Random — fastest clicker wins",    "FIFO — first in line, first served"],
    ["Server stability",     "High risk of crash",               "Stable, controlled load"],
    ["Successful bookings",  "Unpredictable, many retries",      "Clean, one attempt per user"],
    ["Admission control",    "None — everyone floods in",        "Batch-by-batch with TTL"],
]

print("📊 No Queue vs. With Queue\n")
print(tabulate(comparison,
               headers=["Aspect", "💥 No Queue", "✅ With Queue"],
               tablefmt="simple_grid"))

---

## 🌍 Real-World Considerations

Our queue is a simplified version of what Ticketmaster and similar platforms use in production. Here are the real-world enhancements:

### 📡 SSE (Server-Sent Events) for Real-Time Position Updates

In production, users don't refresh the page to check their position. The server pushes updates in real-time using **Server-Sent Events (SSE)**:

```
Server → Browser: "You are #4,521 in line. Estimated wait: 8 minutes."
Server → Browser: "You are #3,200 in line. Estimated wait: 5 minutes."
Server → Browser: "You are #12 in line. Almost there!"
Server → Browser: "You're in! Redirecting to the booking page..."
```

### ⚖️ Queue Fairness

Using timestamps as scores guarantees **first-come, first-served** ordering. Some platforms add randomization during the initial rush (everyone who joins within the first 30 seconds gets a random position) to account for network latency differences.

### ⏰ TTL for Inactive Users

If an admitted user doesn't complete their purchase within the TTL window (e.g., 10 minutes), their admission expires automatically. The system then admits the next user from the queue — no spots are wasted.

### 🚦 Rate Limiting the Queue Join Endpoint

Even the "join queue" endpoint needs protection! Without rate limiting, bots could flood the queue with fake entries. Common approaches:
- **CAPTCHA** before joining
- **IP-based rate limiting** (e.g., 1 join per IP per event)
- **Account verification** (only verified accounts can join)

### 🔄 Horizontal Scaling

Redis sorted sets work perfectly across multiple app servers because Redis is a **centralized** data store. Whether you have 1 server or 100, they all read/write to the same Redis queue. This makes the pattern inherently scalable.

```
App Server 1 ─┐
App Server 2 ─┤──→ Redis (single sorted set) ──→ Consistent queue state
App Server 3 ─┘
```

In [ ]:
# ============================================================
# 🧹 Cleanup
# ============================================================
# Remove all queue-related Redis keys and reset database state.

r = get_redis_client()

# Delete queue keys
deleted_count = 0
for key in r.scan_iter("queue:*"):
    r.delete(key)
    deleted_count += 1
for key in r.scan_iter("admitted:*"):
    r.delete(key)
    deleted_count += 1

print(f"🧹 Deleted {deleted_count} Redis keys")

# Reset database
conn = get_pg_connection()
cur = conn.cursor()
cur.execute("DELETE FROM booking_tickets;")
cur.execute("DELETE FROM bookings;")
cur.execute("UPDATE tickets SET status = 'available' WHERE event_id = 1;")
cur.execute("""
    UPDATE tickets SET status = 'sold'
    WHERE id IN (
        SELECT id FROM tickets WHERE event_id = 1 ORDER BY id LIMIT 25
    );
""")
conn.commit()
cur.close()
conn.close()

print("🧹 Database reset to original state")
print("✅ Cleanup complete!")

---

## 📝 Summary

### What we learned

| Concept | Key Takeaway |
|---------|-------------|
| **The Flash Sale Problem** | Millions of simultaneous requests crush the database and create a terrible user experience |
| **Virtual Waiting Queue** | Put users in a Redis sorted set instead of letting them all hit the database |
| **Redis Sorted Sets** | `ZADD` (join), `ZRANK` (position), `ZPOPMIN` (admit) — all O(log N), even with millions of users |
| **Batch Admission** | Pop N users at a time, give them a TTL to complete their purchase |
| **Protected Booking** | Check admission before allowing database access — only admitted users can book |

### The pattern in one sentence

> **Don't let users fight the database directly — make them wait in a fair, fast, Redis-backed queue and admit them in controlled batches.**

### 🔜 Next Up: Notebook 3 — Payment & Reservation Flow

Now that users can get into the booking page in an orderly way, what happens when they actually try to **pay**? In the next notebook, we'll explore:

- **Seat reservation with TTL** — hold a seat while the user enters payment details
- **Handling payment failures** — what if the payment times out or gets declined?
- **Idempotency** — preventing double charges
- **The reservation → payment → confirmation pipeline**